In [1]:
%load_ext autoreload
%autoreload 2

import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

from pathlib import Path
import sys
_here = Path.cwd().resolve()
_root = next(p for p in [_here, *_here.parents] if (p / "src").is_dir())
sys.path.insert(0, str(_root / "src"))
from regression_modelling.constants import PREDICTOR_COLS, TARGET_CATEGORIES
from regression_modelling.data_wrangling.dataset import build_model_table

In [ ]:
# Build the Chicago and Houston crime tables
sns.set_theme(style="whitegrid")
#start_time = time.time()
hou = build_model_table("houston", refresh=True)
chi = build_model_table("chicago",refresh=True)
atlanta = build_model_table("atlanta",refresh=True)
#end_time = time.time()
#print(f"Time taken to build model tables: {end_time - start_time:.2f} seconds")
tables = {
    "houston": hou, 
    "chicago": chi,
    "atlanta": atlanta}   # already built above

In [ ]:
# Check percentile values of columns
def describe_cols(df, cols):
    d = df[cols].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T
    d["skew"] = df[cols].skew()
    d["n_missing"] = df[cols].isna().sum()
    return d.round(2)

for city, df in tables.items():
    print(f"\n{'='*70}\n{city.upper()} — predictor summary\n{'='*70}")
    display(describe_cols(df, PREDICTOR_COLS))

In [ ]:
# Predictor distributions
def plot_grid(df, cols, title, bins=40, color="steelblue"):
    ncol = 3
    nrow = int(np.ceil(len(cols) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 3.2 * nrow))
    for ax, c in zip(axes.ravel(), cols):
        ax.hist(df[c].dropna(), bins=bins, color=color, edgecolor="white")
        ax.set_title(c); ax.set_ylabel("BGs")
    for ax in axes.ravel()[len(cols):]:
        ax.set_visible(False)
    fig.suptitle(title, fontsize=13, y=1.02); fig.tight_layout(); plt.show()

for city, df in tables.items():
    plot_grid(df, PREDICTOR_COLS, f"{city.title()} — predictor distributions")

In [ ]:
# Target summary
# pct_zero is the key column: a target with very high zero-fraction is a poor
# candidate for OLS on log-count and may need pooling or a count model.
def target_summary(df):
    rows = []
    for c in TARGET_CATEGORIES:
        cnt = df[f"{c}_count"]
        rows.append({
            "target": c, "total": int(cnt.sum()), "mean": round(cnt.mean(), 2),
            "p50": cnt.median(), "p95": cnt.quantile(.95), "max": cnt.max(),
            "pct_zero": round((cnt == 0).mean() * 100, 1), "skew": round(cnt.skew(), 2),
        })
    return pd.DataFrame(rows)

for city, df in tables.items():
    print(f"\n{city.upper()} — target (count) summary")
    display(target_summary(df))

1. **Large pct_zero** values for murder and rape, as one would expect
2. Block groups with Robbery and Burglary now seem in line after data refresh! 

In [ ]:
# Confirms log(count+1) tames the skew that raw counts show — justifying the OLS target.
def plot_target_forms(df, cat, city, rate_clip_pct=99):
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    axes[0].hist(df[f"{cat}_count"], bins=40, color="indianred", edgecolor="white")
    axes[0].set_title(f"{cat}_count (raw)")
    axes[1].hist(df[f"{cat}_logcount"], bins=40, color="seagreen", edgecolor="white")
    axes[1].set_title(f"{cat}_logcount = log(count+1)")

    rate = df[f"{cat}_rate"].dropna()
    rate_range = (0, rate.quantile(rate_clip_pct / 100)) if rate_clip_pct else None
    axes[2].hist(rate, bins=40, range=rate_range, color="slateblue", edgecolor="white")
    axes[2].set_title(f"{cat}_rate (per 1K pop, ≤P{rate_clip_pct})")

    fig.suptitle(f"{city} — {cat}: target forms", y=1.03); fig.tight_layout(); plt.show()

for city, df in tables.items():
    plot_target_forms(df, "cl_total", city.title())

In [ ]:
# Correlation heatmap
def corr_heatmap(df, cols, title):
    corr = df[cols].corr(method="spearman")
    fig, ax = plt.subplots(figsize=(10, 12))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlGn", center=0,
                vmin=-1, vmax=1, square=True, cbar_kws={"shrink": .8}, ax=ax)
    ax.set_title(title); plt.tight_layout(); plt.show()

for city, df in tables.items():
    corr_heatmap(df, PREDICTOR_COLS, f"{city.title()} — predictor correlations (Spearman)")

In [ ]:
df[PREDICTOR_COLS].dtypes

In [ ]:
# VIF calculation

def vif_table(df, cols):
    Xnum = df[cols].apply(pd.to_numeric, errors="coerce").astype("float64")
    X = add_constant(df[cols].fillna(df[cols].median()))
    vif = pd.DataFrame({
        "feature": X.columns,
        "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
    })
    return vif[vif["feature"] != "const"].sort_values("VIF", ascending=False).round(2)

for city, df in tables.items():
    print(f"\n{city.upper()} — VIF")
    display(vif_table(df, PREDICTOR_COLS))

In [ ]:
# Spearman of each predictor against each target (log-count). Directional preview
# of which predictors carry signal before fitting.
def predictor_target_corr(df, targets, suffix):
    tcols = [f"{t}_{suffix}" for t in targets]
    corr = df[PREDICTOR_COLS + tcols].corr(method="spearman").loc[PREDICTOR_COLS, tcols]
    corr.columns = targets
    return corr.round(2)

for city, df in tables.items():
    print(f"\n{city.upper()} — predictor vs target (Spearman, log-count)")
    display(predictor_target_corr(df, TARGET_CATEGORIES, "logcount"))

In [ ]:
# Correlations with rate
suffix="rate"
for city, df in tables.items():
    print(f"\n{city.upper()} — predictor vs target (Spearman, {suffix})")
    display(predictor_target_corr(df, TARGET_CATEGORIES, f"{suffix}"))